# Entregável 4 — Limpeza e Correção dos Dados (Data Cleaning)

**Disciplina:** Aquisição de Biossinais  
**Equipe:** José Lessa & Matheus Rocha  
**Orientador:** Prof. Dr. Victor Hugo C. de Albuquerque  
**Dataset:** PTB-XL (PhysioNet)  

---

## Objetivo

Aplicar técnicas de processamento digital de sinais para eliminar artefatos biológicos e 
instrumentais preservando a morfologia clínica do ECG. O objetivo não é "apagar dados", 
mas corrigir interferências como flutuação de linha de base (baseline wander), ruído da 
rede elétrica e artefatos musculares.

### Conteúdo abordado:
- **Filtro Notch:** Remoção da interferência de rede em 50 Hz
- **Filtro Passa-Banda (Band-pass):** 0.5 Hz a 40 Hz
- **Tratamento de Outliers:** Identificação via MAD (Mediana do Desvio Absoluto) e Winsorization
- **Validação Estatística:** Comparação do Sinal-Ruído (SNR) antes e depois via teste de Wilcoxon e Effect Size (Cohen's d)


## 1. Configuração e Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb
from pathlib import Path
from scipy import stats, signal as sp_signal
import warnings
warnings.filterwarnings('ignore')

# Configurações de plot
plt.rcParams['figure.figsize'] = (16, 8)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Diretórios
DATA_DIR = Path('../../data/ptb-xl')
FIG_DIR = Path('../figuras')
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Configuração concluída.')

## 2. Carregamento do Sinal e Justificativa Fisiológica dos Filtros

In [ ]:
def carregar_ecg(ecg_id, df, data_dir, sampling_rate=500):
    """Carrega um registro de ECG do PTB-XL."""
    col = 'filename_hr' if sampling_rate == 500 else 'filename_lr'
    filename = df.loc[ecg_id, col]
    record = wfdb.rdrecord(str(data_dir / filename))
    return record

df_meta = pd.read_csv(DATA_DIR / 'ptbxl_database.csv', index_col='ecg_id')

# Selecionar um registro com ruído perceptível para demonstração didática
# O registro 6 (V1, V2 etc.) ou qualquer outro da base com drift servirão
ecg_teste_id = df_meta.index[5]  
record = carregar_ecg(ecg_teste_id, df_meta, DATA_DIR, sampling_rate=500)
fs = record.fs

canal_idx = record.sig_name.index('V1')
sinal_bruto = record.p_signal[:, canal_idx]
tempo = np.arange(len(sinal_bruto)) / fs

print(f'Sinal carregado: Registro #{ecg_teste_id}, Derivação V1, FS={fs} Hz')

### 2.1 Justificativa Fisiológica das Escolhas de Filtragem

1. **Filtro Notch (50 Hz):** A Europa (origem do PTB-XL) utiliza corrente alternada de 50 Hz. Esse ruído contamina o ECG como uma senoide ininterrupta. Um filtro *notch* (rejeita-faixa) estreito foi projetado para abater precisamente essa frequência, preservando o QRS.
2. **Filtro Passa-Alta (0.5 Hz):** Projetado para remover o *baseline wander* (flutuação de linha de base), causado primariamente pela respiração do paciente e movimentos torácicos de baixa frequência. A AHA (American Heart Association) recomenda cortes em torno de 0.05 a 0.5 Hz para preservar o segmento ST.
3. **Filtro Passa-Baixa (40 Hz):** Remove artefatos de eletromiografia (EMG/músculo) e ruídos térmicos de alta frequência, mantendo intactas as durações e as amplitudes clínicas dos complexos PQRST.
4. **Filtro de Fase Linear (Filt-Filt):** Os filtros IIR de Butterworth serão aplicados nos dois sentidos (forward e backward) pela função `filtfilt` da SciPy. Isso garante **distorção de fase zero**, ou seja, os picos (P, R, T) não mudarão de posição no eixo temporal.

## 3. Implementação dos Filtros (Notch e Band-pass)

In [ ]:
def aplicar_notch(sinal, fs, f0=50.0, Q=30.0):
    """
    Aplica um filtro rejeita-faixa (Notch) IIR para remover interferência de rede.
    f0: frequência a ser removida (50 Hz para Europa)
    Q: Fator de qualidade (largura da banda rejeitada)
    """
    b, a = sp_signal.iirnotch(f0, Q, fs)
    sinal_filtrado = sp_signal.filtfilt(b, a, sinal)  # Zero-phase
    return sinal_filtrado

def aplicar_bandpass(sinal, fs, lowcut=0.5, highcut=40.0, ordem=4):
    """
    Aplica um filtro Butterworth passa-banda (Band-pass) IIR.
    """
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = sp_signal.butter(ordem, [low, high], btype='band')
    sinal_filtrado = sp_signal.filtfilt(b, a, sinal)  # Zero-phase
    return sinal_filtrado

def limpar_sinal_ecg(sinal, fs):
    """Pipeline completo de filtragem em frequência para 1 array de ECG."""
    sinal = aplicar_notch(sinal, fs, f0=50.0)
    sinal = aplicar_bandpass(sinal, fs, lowcut=0.5, highcut=40.0)
    return sinal

# Aplicar filtros passo-a-passo
sinal_notch = aplicar_notch(sinal_bruto, fs)
sinal_filtrado = aplicar_bandpass(sinal_notch, fs)

print('Filtragem concluída.')

In [ ]:
# Demonstrar a filtragem no domínio do tempo e frequência
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# TEMPO: Visão global do baseline
axes[0, 0].plot(tempo, sinal_bruto, color='gray', alpha=0.5, label='Bruto (com drift)')
axes[0, 0].plot(tempo, sinal_filtrado, color='indigo', linewidth=1.2, label='Filtrado (0.5-40Hz + Notch)')
axes[0, 0].set_title('Correção de Flutuação de Linha de Base (Tempo Total)', fontweight='bold')
axes[0, 0].set_xlabel('Tempo (s)')
axes[0, 0].set_ylabel('Amplitude (mV)')
axes[0, 0].legend()

# TEMPO: Zoom ilustrando preservação morfológica
trecho_inicio, trecho_fim = int(3 * fs), int(5 * fs) # 3s a 5s
axes[0, 1].plot(tempo[trecho_inicio:trecho_fim], sinal_bruto[trecho_inicio:trecho_fim], 
                color='gray', alpha=0.6, label='Bruto')
axes[0, 1].plot(tempo[trecho_inicio:trecho_fim], sinal_filtrado[trecho_inicio:trecho_fim], 
                color='indigo', linewidth=1.5, label='Filtrado')
axes[0, 1].set_title('Zoom Preservação Morfológica (Janela 3-5s)', fontweight='bold')
axes[0, 1].set_xlabel('Tempo (s)')
axes[0, 1].set_ylabel('Amplitude (mV)')
axes[0, 1].legend()

# FREQUÊNCIA: Espectro de Potência (PSD)
f_bruto, Pxx_bruto = sp_signal.welch(sinal_bruto, fs, nperseg=4096)
f_notch, Pxx_notch = sp_signal.welch(sinal_notch, fs, nperseg=4096)
f_filt, Pxx_filt = sp_signal.welch(sinal_filtrado, fs, nperseg=4096)

# Focus em 0 a 100 Hz
l_idx = np.where(f_bruto <= 100)[0]

axes[1, 0].semilogy(f_bruto[l_idx], Pxx_bruto[l_idx], color='gray', alpha=0.5, label='Sinal Bruto')
axes[1, 0].semilogy(f_bruto[l_idx], Pxx_filt[l_idx], color='indigo', linewidth=1.2, label='Após Band-pass (0.5-40Hz)')
axes[1, 0].set_title('Espectro de Potência: Efeito do Filtro Passa-Banda', fontweight='bold')
axes[1, 0].set_xlabel('Frequência (Hz)')
axes[1, 0].set_ylabel('PSD')
axes[1, 0].legend()

axes[1, 1].semilogy(f_bruto[l_idx], Pxx_bruto[l_idx], color='gray', alpha=0.5, label='Sinal Bruto')
axes[1, 1].semilogy(f_notch[l_idx], Pxx_notch[l_idx], color='#e67e22', linewidth=1.2, label='Após Notch (50Hz)')
axes[1, 1].set_title('Espectro de Potência: Efeito do Filtro Notch em 50Hz', fontweight='bold')
axes[1, 1].set_xlabel('Frequência (Hz)')
axes[1, 1].set_xlim(40, 60)
axes[1, 1].set_ylabel('PSD')
axes[1, 1].legend()

plt.suptitle(f'Efeitos dos Filtros Projetados (Registro {ecg_teste_id})', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(FIG_DIR / 'efeito_das_filtragens_tempo_frequencia.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

## 4. Tratamento de Outliers Extremos Transientes

Sinais biomédicos frequentemente contêm *spikes* (picos anômalos) devido à movimentação súbita de cabos, contrações musculares isoladas ou choques mecânicos. 

### Estratégia Adotada
1. **Métrica (MAD):** Conforme demonstrado no Entregável 3, ECG não possui distribuição Normal ideal. Portanto, rejeição via Z-Score é inapropriada, pois a média e desvio padrão seriam enviesados pelo próprio *spike*. Usaremos o **Median Absolute Deviation (MAD)**, uma métrica de desvio extremamente robusta a outliers.
2. **Ação (Winsorization + Interpolação):** Não podemos simplesmente "deletar" amostras do sinal, pois isso quebra a continuidade da série temporal (e arruína os filtros digitais que dependem do histórico das amostras no tempo). Aplicaremos *Winsorization*: seletivamente travar ("clipar") as amostras que excedem $K \times MAD$ num limiar razoável e, se necessário, suavizar o segmento em torno via Splines para assemelhar à continuidade fisiológica.

In [ ]:
def detectar_outliers_mad(sinal, thresh=8.0):
    """
    Detecta outliers baseando-se no desvio absoluto da mediana (MAD).
    Um limiar típico em sinais com picos naturais (QRS) é entre 6 e 10.
    """
    mediana = np.median(sinal)
    diff = np.abs(sinal - mediana)
    mad = np.median(diff)
    
    if mad == 0: # Caso extremo ou flat
        return np.zeros_like(sinal, dtype=bool)
        
    z_mad = (0.6745 * diff) / mad
    outliers_mask = z_mad > thresh
    return outliers_mask

def aplicar_winsorization_interpolacao(sinal, outliers_mask):
    """
    Aplica interpolação linear nos pontos identificados como outliers 
    para substituir spikes instrumentais por uma curva transicional limpa.
    """
    sinal_limpo = sinal.copy()
    
    if not np.any(outliers_mask):
        return sinal_limpo
        
    # Obter os índices normais (não-outliers)
    x_normais = np.where(~outliers_mask)[0]
    y_normais = sinal_limpo[x_normais]
    x_outliers = np.where(outliers_mask)[0]
    
    # Interpolar nos pontos de outlier usando dados vizinhos normais
    y_interpolado = np.interp(x_outliers, x_normais, y_normais)
    sinal_limpo[x_outliers] = y_interpolado
    
    return sinal_limpo

# Simular um spike extremo de cabo solto no sinal filtrado apenas para visualização didática
np.random.seed(42)
sinal_com_artefato = sinal_filtrado.copy()
indice_spike = int(6.5 * fs)
sinal_com_artefato[indice_spike:indice_spike+15] += 5.0  # +5mV é improvavel pro coração (outlier artificial)

# Aplicar tratamento
mascara_outliers = detectar_outliers_mad(sinal_com_artefato, thresh=10.0)
sinal_recuperado = aplicar_winsorization_interpolacao(sinal_com_artefato, mascara_outliers)

print(f"Artefatos identificados: {mascara_outliers.sum()} amostras corrompidas.")

In [ ]:
# Plotar
fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(tempo, sinal_com_artefato, color='#e74c3c', alpha=0.5, linewidth=2, label='Com Artefato (Spike/Outlier)')
ax.plot(tempo, sinal_recuperado, color='#2ecc71', linewidth=1.2, label='Recuperado (MAD + Interpolação)')

# Destacar área onde o MAD atuou
ax.fill_between(tempo, -2, 6, where=mascara_outliers, color='yellow', alpha=0.3, label='Região Reparada')

ax.set_title('Tratamento de Outliers com MAD e Interpolação', fontweight='bold', fontsize=14)
ax.set_xlabel('Tempo (s)')
ax.set_ylabel('Amplitude (mV)')
ax.set_ylim([-2, 6])
ax.legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'tratamento_outliers_mad.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

## 5. Pipeline Integrado de Limpeza Aplicado à População

Agora aplicamos todo o nosso pipeline (Notch + Band-pass + MAD/Interpolação) em 
uma amostra populacional para computar evidências estatísticas. Calcularemos o **Signal-to-Noise Ratio (SNR)** antes e depois da limpeza.

In [ ]:
def calcular_snr(sinal, fs):
    """Estima o SNR assumindo que as componentes 0.5-40Hz são 'sinal' e o resto 'ruído'."""
    nyq = fs / 2
    b, a = sp_signal.butter(4, [0.5/nyq, 40.0/nyq], btype='band')
    sinal_ideal = sp_signal.filtfilt(b, a, sinal)
    ruido = sinal - sinal_ideal
    pot_sinal = np.mean(sinal_ideal**2)
    pot_ruido = np.mean(ruido**2)
    if pot_ruido < 1e-12:
        return 60.0
    return 10 * np.log10(pot_sinal / pot_ruido)

# Executar em 200 registros (para não demorar, usando D2 - lead II como proxy)
N_AMOSTRA = 200
np.random.seed(42)
ids_amostra = np.random.choice(df_meta.index, size=N_AMOSTRA, replace=False)

resultados_limpeza = []

for idx, ecg_id in enumerate(ids_amostra):
    if idx % 50 == 0:
        print(f'Limpando {idx}/{N_AMOSTRA}...')
        
    try:
        record = carregar_ecg(ecg_id, df_meta, DATA_DIR, sampling_rate=500)
        fs = record.fs
        ch_name = 'II'
        if ch_name in record.sig_name:
            ch_idx = record.sig_name.index(ch_name)
            sinal_bruto_atual = record.p_signal[:, ch_idx]
            
            # Aplicar pipeline de limpeza!
            sinal_filtrado = limpar_sinal_ecg(sinal_bruto_atual, fs)
            mascara_mad = detectar_outliers_mad(sinal_filtrado, thresh=15.0)
            sinal_final_limpo = aplicar_winsorization_interpolacao(sinal_filtrado, mascara_mad)
            
            # Métricas Antes vs Depois
            snr_antes = calcular_snr(sinal_bruto_atual, fs)
            snr_depois = calcular_snr(sinal_final_limpo, fs)
            
            resultados_limpeza.append({
                'ecg_id': ecg_id,
                'snr_antes': snr_antes,
                'snr_depois': snr_depois,
                'outliers_removidos': mascara_mad.sum()
            })
    except Exception as e:
        pass

df_limpeza = pd.DataFrame(resultados_limpeza)
print(f"\nPipeline concluído. {len(df_limpeza)} registros comparados.")

## 6. Validação Estatística (Testes Formal de Eficácia)

Para justificar tecnicamente as etapas de limpeza no fluxo, testamos a hipótese de que a limpeza **aumentou o SNR de forma estatisticamente significativa**.

Como demonstrado no Entregável 3, não dependemos da suposição de Normalidade. Optamos pelo **Teste Não-Paramétrico de Wilcoxon (Signed-Rank Test)**, específico para comparar medidas dependentes/pareadas (o *mesmo* paciente, antes e depois).

- **H₀:** A limpeza não alterou a mediana do SNR (M_antes == M_depois)
- **H₁:** O SNR pós-limpeza é significativamente maior
- **Effect Size (Cohen's d pareado):** Vai atestar se a melhoria detectada é trivial ou considerável.

In [ ]:
# Diferenças de SNR
diff = df_limpeza['snr_depois'] - df_limpeza['snr_antes']

# Wilcoxon signed-rank test
stat_w, p_val = stats.wilcoxon(df_limpeza['snr_antes'], df_limpeza['snr_depois'], alternative='less')

# Cohen's d (Estimativa para dados pareados)
cohen_d = diff.mean() / diff.std()

print('=' * 80)
print('TABELA DE VALIDAÇÃO ESTATÍSTICA DA LIMPEZA DOS DADOS (Wilcoxon Pareado)')
print('=' * 80)

tabela_stats = pd.DataFrame({
    'Métrica avaliada': ['Signal-to-Noise Ratio (SNR) em dB'],
    'Média Antes': [f"{df_limpeza['snr_antes'].mean():.2f} dB"],
    'Média Depois': [f"{df_limpeza['snr_depois'].mean():.2f} dB"],
    'Mediana Antes': [f"{df_limpeza['snr_antes'].median():.2f} dB"],
    'Mediana Depois': [f"{df_limpeza['snr_depois'].median():.2f} dB"],
    'Teste p-value': [f"{p_val:.2e} {'(Significativo ***)' if p_val < 0.001 else ''}"],
    "Effect Size (Cohen's d)": [f"{cohen_d:.2f}"]
})

display(tabela_stats.T)

print('\nInterpretação do Effect Size (Cohen): >0.2=Pequeno, >0.5=Médio, >0.8=Grande.')
print('Portanto, a filtragem obteve uma melhoria de classe considerável.')

In [ ]:
# Gráfico Antes x Depois para validar estatisticamente
fig, ax = plt.subplots(figsize=(10, 6))

sns.violinplot(data=[df_limpeza['snr_antes'], df_limpeza['snr_depois']], palette=['gray', 'mediumpurple'], ax=ax, inner="quartile")

# Ligar as linhas para mostrar que é pareado (amostrando 50 senao polui)
amostra_lin = df_limpeza.head(50)
for i in range(len(amostra_lin)):
    ax.plot([0, 1], [amostra_lin['snr_antes'].iloc[i], amostra_lin['snr_depois'].iloc[i]], 
            color='black', alpha=0.1, linewidth=1)
    
ax.set_xticks([0, 1])
ax.set_xticklabels(['SNR Bruto (Antes)', 'SNR Limpo (Depois)'], fontweight='bold', fontsize=12)
ax.set_ylabel('SNR Estimado (dB)', fontsize=12)
ax.set_title(f"Distribuição da Melhoria do Sinal\np={p_val:.1e} | Cohen's d={cohen_d:.2f}", fontweight='bold', fontsize=14)

plt.tight_layout()
plt.savefig(FIG_DIR / 'violin_comparacao_SNR_limpeza.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

In [ ]:
df_limpeza.to_csv(FIG_DIR.parent / 'resultados_estatisticos_limpeza.csv', index=False)
print("Tabela de comparação salva em CSV.")